In [ ]:
import pandas as pd

# ---------- CONFIG ----------
CSV_FILE = "data.csv"                     # your full dataset
TARGET_PER_YEAR = 20_000
YEARS = range(2001, 2026)
PER_MONTH = TARGET_PER_YEAR // 12         # ≈1666
OUTPUT_FILE = "Crime_Data_2001_2025.csv"
# -----------------------------

print("🔄 Reading full dataset...")
df = pd.read_csv(CSV_FILE, parse_dates=["Date"])

# Extract year, month, day
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["day"] = df["Date"].dt.day

# Keep only 2001–2025 range
df = df[df["year"].between(2001, 2025)]

final_samples = []

print("⚙️ Sampling 20,000 rows per year with month and day balance...")

for year in YEARS:
    year_df = df[df["year"] == year]
    if year_df.empty:
        continue

    sampled_months = []

    for month in range(1, 13):
        month_df = year_df[year_df["month"] == month]
        if month_df.empty:
            continue

        # Split month into early/mid/late
        early = month_df[month_df["day"].between(1, 10)]
        mid   = month_df[month_df["day"].between(11, 20)]
        late  = month_df[month_df["day"] >= 21]

        n_each = PER_MONTH // 3  # ≈555 per part
        month_samples = []

        for part_df in [early, mid, late]:
            n = min(n_each, len(part_df))
            if n > 0:
                month_samples.append(part_df.sample(n=n, random_state=42))

        month_sampled = pd.concat(month_samples, ignore_index=True)

        # Fill remainder if short month
        if len(month_sampled) < PER_MONTH:
            remaining = PER_MONTH - len(month_sampled)
            fill = month_df.sample(n=min(remaining, len(month_df)), random_state=99)
            month_sampled = pd.concat([month_sampled, fill], ignore_index=True)

        sampled_months.append(month_sampled)

    # Combine for the year and trim/extend to 20,000
    year_sampled = pd.concat(sampled_months, ignore_index=True)

    if len(year_sampled) > TARGET_PER_YEAR:
        year_sampled = year_sampled.sample(n=TARGET_PER_YEAR, random_state=7)
    elif len(year_sampled) < TARGET_PER_YEAR:
        remaining = TARGET_PER_YEAR - len(year_sampled)
        fill = year_df.sample(n=min(remaining, len(year_df)), random_state=13)
        year_sampled = pd.concat([year_sampled, fill], ignore_index=True)

    final_samples.append(year_sampled)
    print(f"✅ Year {year}: {len(year_sampled)} rows")

# Combine all years and save
final_df = pd.concat(final_samples, ignore_index=True)
final_df.to_csv(OUTPUT_FILE, index=False)

print("\n🎉 Sampling complete!")
print("Total sampled rows:", len(final_df))
print(final_df['year'].value_counts().sort_index())
